# 🌍 Company Universe - Consolidation Complète

Ce notebook génère le **Company Universe consolidé** en fusionnant toutes les sources de données disponibles.

**Sources de données :**
- ✅ **Key Market Data** : Données de marché depuis `extract_key_market_data.ipynb` (500 entreprises S&P 500 avec 10-K)
  - Données S&P 500 (Weight, Price)
  - Métriques financières (Market Cap, Revenue, Net Income, EPS, FCF)
  - Secteurs et industries (yfinance)
  - Ratios financiers calculés (P/E Ratio, etc.)
- ✅ **Data Points 10-K** : Extraction géographie, segments, supply chain (500 entreprises)

**Hypothèse :**
- Les deux sources contiennent **exactement les mêmes 500 entreprises** (S&P 500 avec 10-K)
- Chaque ticker présent dans Key Market Data est également présent dans Data Points 10-K

**Objectif :**
- Fusionner Key Market Data + Data Points 10-K par ticker
- Créer une structure unifiée pour les 500 entreprises


## Étape 1 : Imports et Configuration


In [19]:
import json
import pandas as pd
from pathlib import Path
from typing import Dict, Optional, List
from collections import defaultdict
import os
from tqdm import tqdm

# Configuration des chemins
BASE_DIR = Path('../../')
DATA_DIR = BASE_DIR / 'data' / 'generated'

KEY_MARKET_DATA_FILE = DATA_DIR / 'key_market_data' / 'all_market_data.json'
DATA_POINTS_DIR = DATA_DIR / 'extracted_data_points'
OUTPUT_DIR = DATA_DIR / 'company_universe'

# Créer le dossier de sortie
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Dossier de sortie : {OUTPUT_DIR}")


📁 Dossier de sortie : ..\..\data\generated\company_universe


## Étape 2 : Charger Key Market Data


In [20]:
print("📊 Chargement des Key Market Data depuis all_market_data.json...")
print(f"   Fichier: {KEY_MARKET_DATA_FILE}")

if not KEY_MARKET_DATA_FILE.exists():
    print(f"⚠️ ERREUR: Fichier non trouvé!")
    print(f"   Exécutez d'abord extract_key_market_data.ipynb pour générer ce fichier")
    raise FileNotFoundError(f"Fichier manquant: {KEY_MARKET_DATA_FILE}")

with open(KEY_MARKET_DATA_FILE, 'r', encoding='utf-8') as f:
    key_market_data = json.load(f)

print(f"✅ {len(key_market_data)} entreprises chargées depuis Key Market Data")
print(f"   - Toutes sont dans le S&P 500 et ont un 10-K disponible")

# Statistiques des données
sector_count = sum(1 for v in key_market_data.values() if v.get('sector') and v.get('sector') != 'Unknown')
performance_count = sum(1 for v in key_market_data.values() if v.get('market_cap'))

print(f"\n📊 Détails:")
print(f"   - Avec secteur (yfinance): {sector_count}")
print(f"   - Avec métriques financières: {performance_count}")
print(f"   - Avec prix S&P 500: {sum(1 for v in key_market_data.values() if v.get('current_price'))}")

# Afficher un exemple
example_ticker = 'AAPL'
if example_ticker in key_market_data:
    print(f"\n📋 Exemple ({example_ticker}):")
    example_data = key_market_data[example_ticker]
    print(f"   - Company: {example_data.get('company_name')}")
    print(f"   - Sector: {example_data.get('sector')}")
    print(f"   - Industry: {example_data.get('industry')}")
    print(f"   - S&P 500 Weight: {example_data.get('sp500_weight')}")
    print(f"   - Market Cap: ${example_data.get('market_cap', 0)/1e12:.2f}T")
    print(f"   - Revenue: ${example_data.get('revenue', 0)/1e9:.2f}B")
    print(f"   - P/E Ratio: {example_data.get('pe_ratio')}")
else:
    print(f"\n⚠️ {example_ticker} non trouvé dans les données")


📊 Chargement des Key Market Data depuis all_market_data.json...
   Fichier: ..\..\data\generated\key_market_data\all_market_data.json
✅ 500 entreprises chargées depuis Key Market Data
   - Toutes sont dans le S&P 500 et ont un 10-K disponible

📊 Détails:
   - Avec secteur (yfinance): 499
   - Avec métriques financières: 499
   - Avec prix S&P 500: 500

📋 Exemple (AAPL):
   - Company: Apple Inc.
   - Sector: Technology
   - Industry: Consumer Electronics
   - S&P 500 Weight: 0.0597
   - Market Cap: $3.79T
   - Revenue: $408.62B
   - P/E Ratio: 36.24


## Étape 3 : Charger Data Points 10-K
 

In [21]:
print("📄 Chargement des Data Points 10-K...")

# Lister tous les fichiers data points
data_points_files = list(DATA_POINTS_DIR.glob('*_data_points.json'))
print(f"📁 {len(data_points_files)} fichiers trouvés")

# Charger tous les data points
data_points = {}
for file_path in tqdm(data_points_files, desc="Chargement"):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            ticker = data.get('ticker', '').upper()
            if ticker:
                data_points[ticker] = data
    except Exception as e:
        print(f"⚠️ Erreur lors du chargement de {file_path.name}: {e}")

print(f"\n✅ {len(data_points)} entreprises avec Data Points 10-K")
print(f"   (Attendu: 500 entreprises - mêmes que Key Market Data)")

# Afficher un exemple
if example_ticker in data_points:
    print(f"\n📋 Exemple ({example_ticker}):")
    print(f"   - Type: {data_points[example_ticker].get('company_type', 'N/A')}")
    print(f"   - Segments: {len(data_points[example_ticker].get('segments', []))} segments")
    print(f"   - Pays: {len(data_points[example_ticker].get('geography', {}).get('countries', []))} pays")


📄 Chargement des Data Points 10-K...
📁 500 fichiers trouvés


Chargement: 100%|██████████| 500/500 [00:00<00:00, 12558.02it/s]


✅ 500 entreprises avec Data Points 10-K
   (Attendu: 500 entreprises - mêmes que Key Market Data)

📋 Exemple (AAPL):
   - Type: Technology - Consumer Electronics and Software Services
   - Segments: 5 segments
   - Pays: 15 pays


## Étape 4 : Fusionner les Données par Entreprise


In [22]:
def merge_company_data(
    ticker: str,
    market_data: Optional[Dict],
    data_points: Optional[Dict]
) -> Dict:
    """
    Fusionne les données entreprises (Market Data + Data Points)
    """
    company = {
        'ticker': ticker,
        'data_sources': {
            'has_market_data': market_data is not None,
            'has_data_points': data_points is not None
        },
        'market_data': market_data or {},
        'data_points': data_points or {}
    }
    
    # Extraire le nom de l'entreprise depuis n'importe quelle source
    company_name = (
        market_data.get('company_name') if market_data
        else data_points.get('company_name') if data_points
        else None
    )
    
    if company_name:
        company['company_name'] = company_name
    
    return company

# Obtenir tous les tickers uniques
# BASE: Utiliser Key Market Data comme source principale (500 entreprises S&P 500 avec 10-K)
all_tickers = set(key_market_data.keys())

# Ajouter les tickers avec Data Points 10-K (certains peuvent ne pas être dans Key Market Data)
all_tickers.update(data_points.keys())

print(f"📊 Total de {len(all_tickers)} entreprises uniques à traiter")
print(f"   - Depuis Key Market Data: {len(key_market_data)}")
print(f"   - Avec Data Points 10-K: {len(data_points)}")
print(f"   - Intersection (les deux sources): {len(set(key_market_data.keys()) & set(data_points.keys()))}")
print(f"   - Uniquement dans Key Market Data: {len(set(key_market_data.keys()) - set(data_points.keys()))}")
print(f"   - Uniquement dans Data Points: {len(set(data_points.keys()) - set(key_market_data.keys()))}")


📊 Total de 500 entreprises uniques à traiter
   - Depuis Key Market Data: 500
   - Avec Data Points 10-K: 500
   - Intersection (les deux sources): 500
   - Uniquement dans Key Market Data: 0
   - Uniquement dans Data Points: 0


## Étape 6 : Générer le Company Universe


In [23]:
print("🌍 Génération du Company Universe...")
print(f"   Base: {len(all_tickers)} entreprises (les mêmes dans Key Market Data et Data Points 10-K)")

company_universe = {}
stats = {
    'total_companies': len(all_tickers),
    'with_market_data': 0,
    'with_data_points': 0,
    'with_market_and_dp': 0  # Market Data + Data Points
}

# Parcourir toutes les entreprises (les deux sources ont les mêmes tickers)
for ticker in tqdm(all_tickers, desc="Fusion"):
    # Les deux sources contiennent les mêmes entreprises, donc on peut accéder directement
    market_data = key_market_data.get(ticker)
    dp_data = data_points.get(ticker)
    
    # Mettre à jour les statistiques
    if market_data:
        stats['with_market_data'] += 1
    if dp_data:
        stats['with_data_points'] += 1
    if market_data and dp_data:
        stats['with_market_and_dp'] += 1
    
    # Fusionner les données (Market Data + Data Points)
    company_universe[ticker] = merge_company_data(ticker, market_data, dp_data)

print(f"\n✅ Company Universe généré avec {len(company_universe)} entreprises")
print(f"\n📊 Statistiques:")
print(f"   - Total entreprises: {stats['total_companies']}")
print(f"   - Avec Market Data: {stats['with_market_data']} ({stats['with_market_data']/stats['total_companies']*100:.1f}%)")
print(f"   - Avec Data Points 10-K: {stats['with_data_points']} ({stats['with_data_points']/stats['total_companies']*100:.1f}%)")
print(f"   - Avec Market Data + Data Points: {stats['with_market_and_dp']} ({stats['with_market_and_dp']/stats['total_companies']*100:.1f}%)")

# Avertissement si toutes les entreprises n'ont pas les deux sources
if stats['with_market_and_dp'] < stats['total_companies']:
    missing_both = stats['total_companies'] - stats['with_market_and_dp']
    print(f"\n⚠️ {missing_both} entreprises n'ont pas les deux sources de données")


🌍 Génération du Company Universe...
   Base: 500 entreprises (les mêmes dans Key Market Data et Data Points 10-K)


Fusion: 100%|██████████| 500/500 [00:00<00:00, 489074.63it/s]


✅ Company Universe généré avec 500 entreprises

📊 Statistiques:
   - Total entreprises: 500
   - Avec Market Data: 500 (100.0%)
   - Avec Data Points 10-K: 500 (100.0%)
   - Avec Market Data + Data Points: 500 (100.0%)


## Étape 7 : Sauvegarder le Company Universe


In [24]:
# Sauvegarder en JSON complet (AVANT enrichissement, sera mis à jour après si enrichissement activé)
output_file_json = OUTPUT_DIR / 'company_universe.json'
with open(output_file_json, 'w', encoding='utf-8') as f:
    json.dump(company_universe, f, indent=2, ensure_ascii=False)

print(f"✅ Fichier JSON sauvegardé: {output_file_json}")
if output_file_json.exists():
    file_size_mb = output_file_json.stat().st_size / 1024 / 1024
    print(f"   Taille: {file_size_mb:.2f} MB")
    print(f"   Entreprises: {len(company_universe)}")
    print("💡 Note: Si enrichissement activé, ce fichier sera mis à jour après")


✅ Fichier JSON sauvegardé: ..\..\data\generated\company_universe\company_universe.json
   Taille: 2.98 MB
   Entreprises: 500
💡 Note: Si enrichissement activé, ce fichier sera mis à jour après


## Étape 8 : Exemple d'Entreprise Complète


In [ ]:
# Afficher un exemple d'entreprise avec le plus de donnees disponibles
example_companies = []
for ticker, company in company_universe.items():
    src = company['data_sources']
    score = sum([src['has_market_data'], src['has_data_points']])
    if score >= 2:  # Au moins 2 sources (Market + Data Points)
        example_companies.append((ticker, company, score))

# Trier par score decroissant
example_companies.sort(key=lambda x: x[2], reverse=True)

if example_companies:
    best_ticker, best_company, score = example_companies[0]
    print(f"Exemple d'entreprise avec donnees completes: {best_ticker}")
    print(f"   Score de completude: {score}/2 (Market Data + Data Points)")
    
    # Sauvegarder dans un fichier pour voir toutes les donnees (sans limite Jupyter)
    import sys
    from pathlib import Path
    
    example_file = OUTPUT_DIR / f'{best_ticker}_example_complete.json'
    with open(example_file, 'w', encoding='utf-8') as f:
        json.dump(best_company, f, indent=2, ensure_ascii=False)
    
    print(f"\nStructure complete sauvegardee dans: {example_file}")
    print("\nAffichage des donnees (peut etre tronque par Jupyter):")
    
    # Afficher quand meme dans le notebook (peut etre limite par Jupyter)
    json_str = json.dumps(best_company, indent=2, ensure_ascii=False)
    print(json_str)
else:
    print("Aucune entreprise avec au moins 2 sources de donnees")


Exemple d'entreprise avec donnees completes: DOW
   Score de completude: 2/2 (Market Data + Data Points)

Structure complete (toutes les donnees):
{
  "ticker": "DOW",
  "data_sources": {
    "has_market_data": true,
    "has_data_points": true
  },
  "market_data": {
    "ticker": "DOW",
    "company_name": "Dow Inc.",
    "sector": "Basic Materials",
    "industry": "Chemicals",
    "is_sp500": true,
    "sp500_weight": 0.0003,
    "current_price": 23.31,
    "market_cap": 16275061707.0,
    "revenue": 41819000000.0,
    "operating_income": 289000000.0,
    "net_income": -981000000.0,
    "eps": -1.4,
    "free_cash_flow": -1790000000.0,
    "profit_margin": -0.0235,
    "operating_margin": 0.0069,
    "pe_ratio": 7.67,
    "pe_ratio_source": "yfinance",
    "price_to_sales": 0.39,
    "fcf_margin": -0.0428,
    "fcf_yield": -0.11,
    "earnings_yield": -0.0601,
    "market_cap_to_earnings": -16.59,
    "operating_efficiency": -0.29,
    "cash_conversion": 1.82
  },
  "data_points": 